<a href="https://www.kaggle.com/code/insanejsk/dense-retriever-train?scriptVersionId=327155501" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install transformers datasets sentence-transformers safetensors -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.3 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import json
import numpy as np
from tqdm import tqdm
from safetensors.torch import load_file, save_file

CONFIG = {
    "model_name"  : "bert-base-uncased",
    "proj_dim"    : 256,
    "max_length"  : 180,       # max tokens per passage
    "batch_size"  : 32,
    "epochs"      : 3,
    "lr"          : 2e-5,
    "temperature" : 0.07,      # InfoNCE temperature — suggested in paper
    "device"      : "cuda" if torch.cuda.is_available() else "cpu",
}

print(f"Device: {CONFIG['device']}")
print(f"GPU: {torch.cuda.get_device_name(0) if CONFIG['device'] == 'cuda' else 'None'}")

Device: cuda
GPU: Tesla T4


In [3]:
class TriplesDataset(Dataset):
    def __init__(self, path):
        with open(path) as f:
            self.data = [json.loads(line) for line in f]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return (
            self.data[idx]["query"],
            self.data[idx]["positive"],
            self.data[idx]["negative"],   # not used in InfoNCE directly
        )                                  # but kept for future versions, if needed

train_dataset = TriplesDataset("/kaggle/input/datasets/insanejsk/nse-dr/triples_train.jsonl")
val_dataset   = TriplesDataset("/kaggle/input/datasets/insanejsk/nse-dr/triples_val.jsonl")

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val samples  : {len(val_dataset)}")

Train batches: 6250
Val samples  : 5000


In [4]:
# Zero-shot BERT-base
def encode_texts(texts, tokenizer, model, device, batch_size=64, max_length=180):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            out = model(**encoded)
        mask = encoded["attention_mask"]
        mask_expanded = mask.unsqueeze(-1).float()
        emb = (out.last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)
        emb = F.normalize(emb, p=2, dim=-1)
        all_embs.append(emb.cpu())
    return torch.cat(all_embs, dim=0)

def evaluate_full_corpus(model_path, val_path, device, sample_queries=500):
    """
    - Corpus  = ALL unique passages in val set (~9000 passages)
    - Queries = sample_queries random queries
    - Each query retrieves from the full corpus
    """
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModel.from_pretrained(model_path).to(device).eval()

    with open(val_path) as f:
        val_data = [json.loads(line) for line in f]

    # Build full corpus
    corpus = list({t["positive"] for t in val_data} |
                  {t["negative"] for t in val_data})
    print(f"Full corpus size: {len(corpus)} passages")

    # query-positive mapping
    pos_map = {t["query"]: t["positive"] for t in val_data}

    # Sample queries
    np.random.seed(42)
    sampled   = [val_data[i] for i in
                 np.random.choice(len(val_data), sample_queries, replace=False)]
    queries   = [s["query"]    for s in sampled]
    positives = [s["positive"] for s in sampled]

    # Map positive text to index
    corpus_idx = {p: i for i, p in enumerate(corpus)}
    pos_indices = [corpus_idx[p] for p in positives]

    # Encode
    print("Encoding corpus...")
    corpus_embs = encode_texts(corpus,  tokenizer, model, device)

    print("Encoding queries...")
    query_embs  = encode_texts(queries, tokenizer, model, device)

    # Score (N_queries, N_corpus)
    scores = torch.matmul(query_embs, corpus_embs.T).numpy()

    recalls, mrrs = [], []
    for i, pos_idx in enumerate(pos_indices):
        ranked = np.argsort(scores[i])[::-1].tolist()
        recalls.append(int(pos_idx in ranked[:10]))
        rank = ranked.index(pos_idx) + 1 if pos_idx in ranked else None
        mrrs.append(1.0 / rank if rank else 0.0)

    print(f"\nRecall@10 : {np.mean(recalls):.4f}")
    print(f"MRR       : {np.mean(mrrs):.4f}")
    return np.mean(recalls), np.mean(mrrs)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("=== Zero-shot BERT-base (full corpus) ===")
evaluate_full_corpus(
    "bert-base-uncased",
    "/kaggle/input/datasets/insanejsk/nse-dr/triples_val.jsonl",
    device
)

=== Zero-shot BERT-base (full corpus) ===


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Encoding: 100%|██████████| 156/156 [01:32<00:00,  1.69it/s]


Encoding queries...


Encoding: 100%|██████████| 8/8 [00:00<00:00, 12.63it/s]



Recall@10 : 0.4060
MRR       : 0.2739


(np.float64(0.406), np.float64(0.2738575194145326))

In [26]:
class BiEncoder(nn.Module):
    def __init__(self, model_name, max_length, proj_dim=256, saved_proj = False):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.encoder   = AutoModel.from_pretrained(model_name)
        self.max_length = max_length
        hidden_size = self.encoder.config.hidden_size  # 768 for BERT-base
        self.proj = nn.Sequential(
            nn.Linear(hidden_size, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
        if saved_proj:
            proj_state = load_file(model_name + "/proj.safetensors")
            self.proj.load_state_dict(proj_state)

    def mean_pool(self, token_embeddings, attention_mask):
        """
        Mean pooling: average token embeddings, ignoring padding tokens.
        """
        mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_embeddings = (token_embeddings * mask_expanded).sum(dim=1)
        sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
        return sum_embeddings / sum_mask

    def encode(self, texts, device):
        """
        Tokenize + encode a list of strings, L2-normalized embeddings.
        """
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        ).to(device)

        outputs = self.encoder(**encoded)
        embeddings = self.mean_pool(
            outputs.last_hidden_state,
            encoded["attention_mask"]
        )
        embeddings = self.proj(embeddings)
        return F.normalize(embeddings, p=2, dim=-1)  # L2 normalize

    def forward(self, queries, positives, device):
        q_emb = self.encode(queries, device)   # (B, D)
        p_emb = self.encode(positives, device)   # (B, D)
        return q_emb, p_emb

In [6]:
def infonce_loss(q_emb, p_emb, temperature):
    """
    InfoNCE loss with in-batch negatives.
    q_emb : (B, D) — query embeddings
    p_emb : (B, D) — positive passage embeddings
    For each query i:
      - positive  = p_emb[i]
      - negatives = p_emb[j] for all j != i  (from same batch)
    Similarity matrix: (B, B)
      sim[i][j] = cosine_sim(q_emb[i], p_emb[j])
    Loss: cross-entropy where correct class = diagonal (i == j)
    """
    # already normalized so dot product = cosine
    sim_matrix = torch.matmul(q_emb, p_emb.T) / temperature

    # Labels: for query i, the positive is at index i
    labels = torch.arange(sim_matrix.size(0)).to(q_emb.device)

    # Cross-entropy over rows
    loss = F.cross_entropy(sim_matrix, labels)
    return loss

In [7]:
def evaluate(model, val_dataset, device, sample_size=1000):
    model.eval()

    all_positives = [val_dataset[i][1] for i in range(len(val_dataset))]
    all_negatives = [val_dataset[i][2] for i in range(len(val_dataset))]
    corpus        = list(set(all_positives + all_negatives))
    corpus_idx    = {p: i for i, p in enumerate(corpus)}

    indices   = np.random.choice(len(val_dataset), sample_size, replace=False)
    samples   = [val_dataset[i] for i in indices]
    queries   = [s[0] for s in samples]
    positives = [s[1] for s in samples]

    pos_indices = [corpus_idx[p] for p in positives]

    with torch.no_grad():
        corpus_embs = []
        for i in range(0, len(corpus), 64):
            batch = corpus[i:i+64]
            emb   = model.encode(batch, device)
            corpus_embs.append(emb.cpu())
        corpus_embs = torch.cat(corpus_embs, dim=0)  # (~9944, D)

        query_embs = []
        for i in range(0, len(queries), 64):
            batch = queries[i:i+64]
            emb   = model.encode(batch, device)
            query_embs.append(emb.cpu())
        query_embs = torch.cat(query_embs, dim=0)    # (1000, D)

    scores = torch.matmul(query_embs, corpus_embs.T).numpy()  # (1000, ~9944)

    recalls, mrrs = [], []
    for i, pos_idx in enumerate(pos_indices):
        ranked = np.argsort(scores[i])[::-1].tolist()
        recalls.append(int(pos_idx in ranked[:10]))
        rank = ranked.index(pos_idx) + 1 if pos_idx in ranked else None
        mrrs.append(1.0 / rank if rank else 0.0)

    model.train()
    return np.mean(recalls), np.mean(mrrs)

In [8]:
# Training block
model = BiEncoder(
    CONFIG["model_name"], 
    CONFIG["max_length"], 
    CONFIG["proj_dim"]
).to(CONFIG["device"])

# Use both T4s if available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
scaler = torch.amp.GradScaler('cuda')

best_recall = 0.0
history     = []

for epoch in range(CONFIG["epochs"]):
    model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")

    for batch_idx, (queries, positives, _) in enumerate(loop):

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):   # mixed precision forward pass
            # Handle DataParallel — call module directly
            enc = model.module if hasattr(model, "module") else model
            q_emb, p_emb = enc(queries, positives, CONFIG["device"])
            loss = infonce_loss(q_emb, p_emb, CONFIG["temperature"])

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    # Evaluate at end of each epoch
    enc = model.module if hasattr(model, "module") else model
    recall, mrr = evaluate(enc, val_dataset, CONFIG["device"])

    history.append({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "recall@10": recall,
        "mrr": mrr,
    })

    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f} | "
          f"Recall@10: {recall:.4f} | MRR: {mrr:.4f}")

    # Saving best model
    if recall > best_recall:
        best_recall = recall
        enc = model.module if hasattr(model, "module") else model
        enc.encoder.save_pretrained("best_biencoder")
        enc.tokenizer.save_pretrained("best_biencoder")
        save_file(
            {k: v for k, v in enc.proj.state_dict().items()},
            "best_biencoder/proj.safetensors",
        )
        print(f"New best saved (Recall@10: {recall:.4f})")

print("=== Training Complete ===")
print(f"Best Recall@10: {best_recall:.4f}")
print(f"BM25 baseline : 0.7714")
print(f"Delta         : {best_recall - 0.7714:+.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using 2 GPUs


Epoch 1/3: 100%|██████████| 6250/6250 [30:16<00:00,  3.44it/s, loss=0.0705]



Epoch 1 | Loss: 0.0902 | Recall@10: 0.9660 | MRR: 0.8223


...


Epoch 2 | Loss: 0.0300 | Recall@10: 0.9760 | MRR: 0.8120


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New best saved (Recall@10: 0.9760)


Epoch 3/3: 100%|██████████| 6250/6250 [30:15<00:00,  3.44it/s, loss=0.0086]



Epoch 3 | Loss: 0.0178 | Recall@10: 0.9690 | MRR: 0.8276
=== Training Complete ===
Best Recall@10: 0.9760
BM25 baseline : 0.7714
Delta         : +0.2046


### Dense Retriever v2
- We'll be using dense hard negatives replacing the previous negatives
- We'll use the trained model to find the closest matches to the true positive
- This will make it harder for the model to recognize the negatives
- This method was illustrated in the ANCE paper, a follow up on the previously mentioned InfoNCE paper

In [20]:
# Dense hard negative mining using trained bi-encoder
model = BiEncoder(
    "/kaggle/working/best_biencoder",
    max_length=180,
    saved_proj = True,
    proj_dim=256
).to(device)
model.eval()

with open("/kaggle/input/datasets/insanejsk/nse-dr/triples_train.jsonl") as f:
    triples = [json.loads(line) for line in f]

print(f"Loaded {len(triples)} triples")

corpus = list({t["positive"] for t in triples} |
              {t["negative"] for t in triples})
corpus_idx = {p: i for i, p in enumerate(corpus)}

print(f"Corpus: {len(corpus)} passages")

# Encode corpus
print("Pre-tokenizing + encoding corpus...")
corpus_embs = []
TOK_BATCH   = 10_000
ENC_BATCH   = 1024

with torch.no_grad():
    for i in tqdm(range(0, len(corpus), TOK_BATCH), desc="Corpus chunks"):
        chunk = corpus[i:i+TOK_BATCH]

        enc = model.tokenizer(
            chunk,
            padding=True,
            truncation=True,
            max_length=180,
            return_tensors="pt"
        )
        ids  = enc["input_ids"]
        mask = enc["attention_mask"]

        for j in range(0, ids.size(0), ENC_BATCH):
            id_b   = ids[j:j+ENC_BATCH].to(device)
            mask_b = mask[j:j+ENC_BATCH].to(device)

            outputs  = model.encoder(input_ids=id_b, attention_mask=mask_b)
            mask_exp = mask_b.unsqueeze(-1).float()
            emb = (outputs.last_hidden_state * mask_exp).sum(1) \
                  / mask_exp.sum(1).clamp(min=1e-9)
            emb = model.proj(emb)
            emb = F.normalize(emb, p=2, dim=-1)
            corpus_embs.append(emb.cpu())

corpus_embs = torch.cat(corpus_embs, dim=0)
print(f"Corpus embeddings: {corpus_embs.shape}")

# Encode queries
print("Pre-tokenizing + encoding queries...")
queries    = [t["query"] for t in triples]
query_embs = []

with torch.no_grad():
    for i in tqdm(range(0, len(queries), TOK_BATCH), desc="Query chunks"):
        chunk = queries[i:i+TOK_BATCH]

        enc = model.tokenizer(
            chunk,
            padding=True,
            truncation=True,
            max_length=180,
            return_tensors="pt"
        )
        ids  = enc["input_ids"]
        mask = enc["attention_mask"]

        for j in range(0, ids.size(0), ENC_BATCH):
            id_b   = ids[j:j+ENC_BATCH].to(device)
            mask_b = mask[j:j+ENC_BATCH].to(device)

            outputs  = model.encoder(input_ids=id_b, attention_mask=mask_b)
            mask_exp = mask_b.unsqueeze(-1).float()
            emb = (outputs.last_hidden_state * mask_exp).sum(1) \
                  / mask_exp.sum(1).clamp(min=1e-9)
            emb = model.proj(emb)
            emb = F.normalize(emb, p=2, dim=-1)
            query_embs.append(emb.cpu())

query_embs = torch.cat(query_embs, dim=0)
print(f"Query embeddings: {query_embs.shape}")

# Mine hard negatives
# Query batches of 512

print("Mining hard negatives...")
hard_triples = []
QUERY_BATCH  = 512
TOP_K        = 20   # retrieve top-20, pick hardest non-positive

corpus_embs_gpu = corpus_embs.to(device)  # move corpus to GPU once

for i in tqdm(range(0, len(triples), QUERY_BATCH), desc="Mining"):
    q_batch    = query_embs[i:i+QUERY_BATCH].to(device)  # (512, 256)
    batch_data = triples[i:i+QUERY_BATCH]

    # (512, 393k) similarity scores
    scores = torch.matmul(q_batch, corpus_embs_gpu.T)

    # Top-K indices for each query in batch
    topk_scores, topk_indices = torch.topk(scores, k=TOP_K+1, dim=-1)
    topk_indices = topk_indices.cpu().numpy()

    for j, triple in enumerate(batch_data):
        pos_idx      = corpus_idx.get(triple["positive"], -1)
        ranked       = topk_indices[j].tolist()

        # Remove positive, take first remaining = hardest negative
        hard_candidates = [corpus[idx] for idx in ranked
                           if idx != pos_idx]

        if not hard_candidates:
            continue

        hard_triples.append({
            "query"   : triple["query"],
            "positive": triple["positive"],
            "negative": hard_candidates[0],
        })

print(f"\nGenerated {len(hard_triples)} hard negative triples")

# Save
out_path = "/kaggle/working/triples_train_hard.jsonl"
with open(out_path, "w") as f:
    for t in hard_triples:
        f.write(json.dumps(t) + "\n")

print(f"Saved to {out_path}")

# Sanity check
ex = hard_triples[100]
print(f"\n--- Example hard negative ---")
print(f"Query    : {ex['query']}")
print(f"Positive : {ex['positive'][:120]}...")
print(f"Hard neg : {ex['negative'][:120]}...")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Projection head loaded from /kaggle/working/best_biencoder/proj.safetensors
Loaded 200000 triples
Corpus: 393154 passages
Pre-tokenizing + encoding corpus...


Corpus chunks: 100%|██████████| 40/40 [1:09:29<00:00, 104.24s/it]


Corpus embeddings: torch.Size([393154, 256])
Pre-tokenizing + encoding queries...


Query chunks: 100%|██████████| 20/20 [08:41<00:00, 26.08s/it]


Query embeddings: torch.Size([200000, 256])
Mining hard negatives...


Mining: 100%|██████████| 391/391 [00:22<00:00, 17.16it/s]



Generated 200000 hard negative triples
Saved to /kaggle/working/triples_train_hard.jsonl

--- Example hard negative ---
Query    : process theories of motivation suggest that
Positive : Process theories of motivation. The group of motivational theories that falls under the umbrella category of Process The...
Hard neg : Types of theories and models. Motivation theories can be classified on a number of bases: Natural vs. Rational: based on...


#### PS: Outputs from the block above were added to the dataset to ensure easy use, and dataset was updated and synced before further execution

In [28]:
# Same training, new dataset
# Starting fresh from BERT-base

train_dataset = TriplesDataset("/kaggle/input/datasets/insanejsk/nse-dr/triples_train_hard.jsonl")

model = BiEncoder(
    CONFIG["model_name"], 
    CONFIG["max_length"], 
    CONFIG["proj_dim"]
).to(CONFIG["device"])

# Use both T4s if available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"])
scaler = torch.amp.GradScaler('cuda')

best_recall = 0.0
history     = []

for epoch in range(CONFIG["epochs"]):
    model.train()
    total_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")

    for batch_idx, (queries, positives, _) in enumerate(loop):

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):   # mixed precision forward pass
            # Handle DataParallel — call module directly
            enc = model.module if hasattr(model, "module") else model
            q_emb, p_emb = enc(queries, positives, CONFIG["device"])
            loss = infonce_loss(q_emb, p_emb, CONFIG["temperature"])

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    # Evaluate per epoch
    enc = model.module if hasattr(model, "module") else model
    recall, mrr = evaluate(enc, val_dataset, CONFIG["device"])

    history.append({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "recall@10": recall,
        "mrr": mrr,
    })

    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f} | "
          f"Recall@10: {recall:.4f} | MRR: {mrr:.4f}")

     # Saving best model
    if recall > best_recall:
        best_recall = recall
        enc = model.module if hasattr(model, "module") else model
        enc.encoder.save_pretrained("best_biencoder")
        enc.tokenizer.save_pretrained("best_biencoder")
        save_file(
            {k: v for k, v in enc.proj.state_dict().items()},
            "best_biencoder/proj.safetensors",
        )
        print(f"New best saved (Recall@10: {recall:.4f})")

print("\n=== Training Complete ===")
print(f"Best Recall@10: {best_recall:.4f}")
print(f"BM25 baseline : 0.7714")
print(f"Delta         : {best_recall - 0.7714:+.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using 2 GPUs


Epoch 1/3: 100%|██████████| 6250/6250 [30:20<00:00,  3.43it/s, loss=0.0412]



Epoch 1 | Loss: 0.0894 | Recall@10: 0.9670 | MRR: 0.8179


...

New best saved (Recall@10: 0.9670)


Epoch 2/3: 100%|██████████| 6250/6250 [30:21<00:00,  3.43it/s, loss=0.0134]



Epoch 2 | Loss: 0.0294 | Recall@10: 0.9670 | MRR: 0.8210


Epoch 3/3: 100%|██████████| 6250/6250 [30:21<00:00,  3.43it/s, loss=0.0128]



Epoch 3 | Loss: 0.0173 | Recall@10: 0.9650 | MRR: 0.8166

=== Training Complete ===
Best Recall@10: 0.9670
BM25 baseline : 0.7714
Delta         : +0.1956


### Failure of v2
- Even though ANCE suggests hard negatives help, since the corpus includes many similar statements which could be considered as positive in itself, but doesn't work in tandem.
- Here, hard negatives hurt the model by overfitting on the nuances of those particular options
- It doesn't generalize well enough for the validation set hence the underperformance of v2 w.r.t. v1
- v2(Recall@10:0.9670) < v1(Recall@10:0.9760)
- This observation suggests that random sampling is better for retriever trained on MS Marco
- MS Marco might have some false negatives because of the nature of its data.